# 01 — Téléchargement du corpus (V2 · dununs)

Chantier **V2** : on se concentre uniquement sur les **dununs** des 3 rythmes de la thèse (Manjanin, Maraka, Woloso).

Ce notebook télécharge, depuis le dépôt **OSF** du corpus *IEMP Malian Jembe* (Polak — accès libre, pas d'inscription), seulement ce dont on a besoin :
- les **pistes isolées de dunun** (`D1`, `D2`),
- leurs **annotations** : onsets de référence, grille métrique, positions dans le cycle.

Tout passe par `utils_V2` (`import utils_V2 as u`). Le téléchargement est **idempotent** (il saute ce qui est déjà là).

## 1. Télécharger

In [ ]:
import utils_V2 as u
rapport = u.download_dunun_corpus()   # D1/D2 + annotations des 3 pièces
rapport

## 2. Vérifier l'intégrité

In [ ]:
import pandas as pd
d = u.list_dunun()
print('Pistes de dunun trouvées :', len(d), '\n')
resume = d.groupby('piece').agg(prises=('take','nunique'), pistes=('code','size'))
resume['2e_dunun'] = d[d['code']=='D2'].groupby('piece').size().reindex(u.PIECES).fillna(0).astype(int).values
display(resume)
# durées + nb d'onsets par piste (contrôle qualité)
rows = []
for _, r in d.iterrows():
    y = u.load_track(r['piece'], r['take'], r['code'])
    on = u.load_onsets(r['piece'], r['take'], r['code'])
    rows.append({'piece':r['piece'], 'prise':r['take'], 'code':r['code'],
                 'durée_s':round(len(y)/u.SR,1), 'onsets':0 if on is None else len(on)})
qc = pd.DataFrame(rows)
print('\nContrôle qualité (durée + onsets par piste) :')
display(qc)

## Bilan
- **19 pistes de dunun** (Manjanin ×6, Maraka ×7, Woloso ×6) + annotations, prêtes en local.
- Chaque piste a sa durée et ses onsets de référence → matière propre pour l'analyse.
- **Suite `02_dunun_extraction_V2`** : extraire les frappes, leurs features, leurs positions dans le cycle, et la couverture D1/D2.